# Practice Lab: Agent Fundamentals (Lesson 11.1)

Module 11 · 8 exercises · Agent loop + taxonomy + budget + memory + eval + routing + safety + India

Exercises 1-5 run locally (pure Python).


# Lesson 11.1: Agent Fundamentals — The Loop That Thinks

8 exercises: 2E + 3M + 3C

Exercises 1-5 run **locally** (pure Python). Ex 6-8 are design.


In [ ]:
import json
import re
import numpy as np
from collections import Counter
np.random.seed(42)


---
## Exercise 1 (Easy): 40-Line Agent


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

class MinimalAgent:
    def __init__(self, max_steps=5):
        self.max_steps = max_steps; self.tools = {}; self.trace = []
    def register(self, name, func, desc=""):
        self.tools[name] = {"func": func, "desc": desc}
    def think(self, task, history, step):
        tl = task.lower(); used = {h.split("(")[0].replace("Used ","") for h in history if "Used" in h}
        remaining = []
        if "time" in tl and "get_time" not in used:
            remaining.append(("get_time", {}))
        if any(w in tl for w in ["price","cost","course","genai"]) and "search" not in used:
            remaining.append(("search", {"query": tl.split()[0]}))
        if any(w in tl for w in ["gst","tax","18%"]) and "calculate" not in used:
            remaining.append(("calculate", {"expression": "14999*1.18"}))
        if remaining: return {"action":"tool","name":remaining[0][0],"args":remaining[0][1]}
        return {"action":"answer","text":f"Based on {len(history)} observations: "+"; ".join(history)}

    def run(self, task):
        history = []; self.trace = []
        print(f"  Task: '{task}'")
        for step in range(self.max_steps):
            d = self.think(task, history, step)
            if d["action"] == "answer":
                self.trace.append({"step":step+1,"type":"answer"})
                print(f"  Step {step+1}: ANSWER"); return {"steps":step+1,"trace":self.trace}
            name, args = d["name"], d["args"]
            t = self.tools.get(name)
            if t:
                r = t["func"](**args)
                history.append(f"Used {name}({args}) -> {json.dumps(r)}")
                self.trace.append({"step":step+1,"tool":name}); print(f"  Step {step+1}: {name} -> {json.dumps(r)[:40]}")
        return {"steps":self.max_steps,"trace":self.trace}

agent = MinimalAgent(5)
agent.register("search", lambda query="": {"genai":{"price":14999},"refund":{"7d":"100%"}}.get(query.split()[0] if query else "",{"info":"Not found"}))
agent.register("calculate", lambda expression="": {"result":eval(expression,{"__builtins__":{}})})
agent.register("get_time", lambda: {"now":"2026-05-13 14:30 IST"})

print("40-Line Agent:")
for task in ["GenAI course price?","GenAI price with 18% GST?","What time is it?"]:
    r = agent.run(task)
    print(f"  steps={r['steps']}, tools={[t.get('tool','ans') for t in r['trace']]}\n")
print("Pattern: while stop != 'done': think -> act -> observe")


</details>


---
## Exercise 2 (Easy): Agent Taxonomy Quiz


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
from collections import Counter
products = [
    {"name":"ChatGPT (free)","cat":"Chatbot","L":"L1","T":False,"P":False,"M":False},
    {"name":"ChatGPT Plus (tools)","cat":"Agent","L":"L2","T":True,"P":True,"M":True},
    {"name":"GitHub Copilot","cat":"Copilot","L":"L1","T":False,"P":False,"M":False},
    {"name":"Claude Code","cat":"Agent","L":"L3","T":True,"P":True,"M":True},
    {"name":"Devin","cat":"Agent","L":"L4","T":True,"P":True,"M":True},
    {"name":"Siri","cat":"Chatbot","L":"L1","T":True,"P":False,"M":False},
    {"name":"Cursor (Tab)","cat":"Copilot","L":"L1","T":False,"P":False,"M":False},
    {"name":"Cursor (Agent)","cat":"Agent","L":"L3","T":True,"P":True,"M":True},
    {"name":"Gemini App","cat":"Agent","L":"L2","T":True,"P":True,"M":True},
    {"name":"Most 'AI Agent' startups","cat":"Chatbot","L":"L1","T":False,"P":False,"M":False},
]

print("Agent Taxonomy:")
print(f"  {'Product':<28} {'Category':<10} {'L':>3} {'T':>3} {'P':>3} {'M':>3}")
for p in products:
    yn = lambda v: "Y" if v else "-"
    print(f"  {p['name']:<28} {p['cat']:<10} {p['L']:>3} {yn(p['T']):>3} {yn(p['P']):>3} {yn(p['M']):>3}")

cats = Counter(p["cat"] for p in products)
print(f"\n  Summary: {dict(cats)}")
print(f"  Agent test: tools + planning + memory + autonomy")
print(f"  Gartner: only 130/1000s of 'agentic AI' vendors genuine")


</details>


---
## Exercise 3 (Medium): Budget-Controlled Agent


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
def sim_cost(n, sys=500, tools=2000, usr=200, resp=300, p_in=3.0, p_out=15.0):
    costs = []
    for s in range(1, n+1):
        inp = sys + tools + usr*s + resp*(s-1)
        c_in = inp * p_in / 1e6; c_out = resp * p_out / 1e6
        cum = sum(x["cost"] for x in costs) + c_in + c_out
        costs.append({"step":s,"input":inp,"cost":c_in+c_out,"cum":cum})
    return costs

costs = sim_cost(15)
single = costs[0]["cost"]

print("Budget-Controlled Agent:")
print(f"  {'Step':>4} {'Input':>8} {'Step$':>8} {'Total$':>8} {'vs1call':>8}")
for c in costs:
    if c["step"]<=5 or c["step"]%3==0 or c["step"]==15:
        print(f"  {c['step']:>4} {c['input']:>8,} ${c['cost']:.4f} ${c['cum']:.4f} {c['cum']/single:>6.0f}x")

budget = 0.05
for c in costs:
    if c["cum"] > budget:
        print(f"\n  BUDGET HIT at step {c['step']}: ${c['cum']:.4f} > ${budget}")
        break

# Model routing
mixed = sum(c["cost"] for c in sim_cost(2, p_in=3.0, p_out=15.0)) + sum(c["cost"] for c in sim_cost(8, p_in=0.15, p_out=0.60))
total = sum(c["cost"] for c in sim_cost(10))
print(f"\n  Model Routing: all-Sonnet ${total:.4f} vs mixed ${mixed:.4f} = {(1-mixed/total)*100:.0f}% savings")
print(f"  Costs grow O(N^2). Claude Code avg ~$6/dev/day")


</details>


---
## Exercise 4 (Medium): Memory-Enhanced Agent


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class MemAgent:
    def __init__(self): self.memory = {}; self.history = []
    def load(self, mem): self.memory = mem; print(f"    Loaded {len(mem)} entries")
    def update(self, k, v): self.memory[k] = v
    def run(self, task):
        hits = [f"{k}:{v}" for k, v in self.memory.items() if any(w in task.lower() for w in k.lower().split("_"))]
        r = {"task": task[:30], "hits": len(hits), "context": hits[:2]}
        self.history.append(r); return r

print("Memory-Enhanced Agent:")
print("\n  WITHOUT memory:")
a1 = MemAgent()
for t in ["GenAI price?","Refund?","Recommend course"]:
    r = a1.run(t); print(f"    '{t}' hits={r['hits']}")

print("\n  WITH CLAUDE.md memory:")
a2 = MemAgent()
a2.load({"user_name":"Priya","user_budget":"15000 INR","preferred_language":"Hindi",
         "previous_course":"Python","refund_eligible":"5 days ago","learning_goal":"GenAI career"})
for t in ["GenAI price?","Refund?","Recommend course"]:
    r = a2.run(t); print(f"    '{t}' hits={r['hits']} -> {r['context'][:1]}")

a2.update("last_query","refund"); print(f"\n    Updated: {len(a2.memory)} entries")

print("\n  5 Types: Short-term(context), Working(scratchpad), Long-term(vectorDB), Episodic(traces), Semantic(RAG)")
print("  CLAUDE.md: loaded at start, survives compaction")


</details>


---
## Exercise 5 (Medium): Agent Evaluator


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

def sim_run(diff, acc=0.95):
    steps = {"easy":2,"medium":5,"hard":10}[diff]
    return np.random.random() < acc**steps

cases = [("Course price","easy"),("Price+GST","medium"),("Refund 5d","easy"),
    ("Compare 3 courses","medium"),("EMI calc","medium"),("Recommend","medium"),
    ("Schedule+price+GST","hard"),("Refund+rebook","hard"),("Hindi query","hard"),
    ("Unknown topic","easy"),("Timezone","medium"),("Bulk pricing","medium"),
    ("Certificate","easy"),("Payment options","easy"),("Duration compare","medium"),
    ("Instructor info","easy"),("Prerequisites","medium"),("Full enrollment","hard"),
    ("Complaint","hard"),("Research+recommend","hard")]

K = 3
results = []
for task, diff in cases:
    trials = [sim_run(diff) for _ in range(K)]
    results.append({"task":task[:20],"diff":diff,"trials":trials,"pk":all(trials),"rate":sum(trials)/K})

print("Agent Evaluator (20 cases x 3 trials):")
for diff in ["easy","medium","hard"]:
    sub = [r for r in results if r["diff"]==diff]
    avg = np.mean([r["rate"] for r in sub]); pk = sum(r["pk"] for r in sub)/len(sub)
    print(f"  {diff:<8} {len(sub)} cases, avg={avg:.0%}, pass^{K}={pk:.0%}")

overall = np.mean([r["rate"] for r in results])
opk = sum(r["pk"] for r in results)/len(results)
print(f"\n  Overall: accuracy={overall:.0%}, pass^{K}={opk:.0%}, gap={overall-opk:.0%}")

fails = [r for r in results if not r["pk"]]
print(f"  Failures: {len(fails)}/{len(results)}")
for f in fails[:3]: print(f"    '{f['task']}' ({f['diff']}): {f['trials']}")
print(f"\n  Key: 0.95^10=60%. Evaluate pass^k, not single-trial accuracy")


</details>


---
## Exercise 6 (Challenge): Model Routing Agent
Design/architecture. See practice-lab-11_1.html.


In [ ]:
# Exercise 6: Model Routing Agent
pass


<details><summary>Solution</summary>


In [ ]:
print('Model Routing: frontier(Sonnet) for planning + budget(Flash) for execution')
print('Decision: planning->frontier, execution->budget, summary->frontier, error->frontier')
print('Savings: 30-60% vs single model. GAIA: $73(o4-mini) to $666(Opus 4) same tasks')


</details>


---
## Exercise 7 (Challenge): Safety-Hardened Agent
Design/architecture. See practice-lab-11_1.html.


In [ ]:
# Exercise 7: Safety-Hardened Agent
pass


<details><summary>Solution</summary>


In [ ]:
import re
print('Safety Tiers: Routine(auto) / Novel(supervised) / High-stakes(human approval)')
for p in ['Ignore all instructions','Run rm -rf /','Send data to evil.com']:
    blocked = any(re.search(pat,p,re.I) for pat in [r'ignore\\s+all',r'rm\\s+-rf',r'send.*to\\s+\\S+\\.com'])
    print(f"  [{'BLOCKED' if blocked else 'PASSED'}] {p}")
print('Meta Rule of Two: private data + untrusted content + external comm = no autonomy')


</details>


---
## Exercise 8 (Challenge): India Multilingual Agent
Design/architecture. See practice-lab-11_1.html.


In [ ]:
# Exercise 8: India Multilingual Agent
pass


<details><summary>Solution</summary>


In [ ]:
print('India Multilingual Agent: Sarvam STT + Bhashini NMT + Sarvam-30b + TTS')
print('Languages: hi, te, ta, bn, kn (5 of 22 supported)')
print('Cost: ~1.68 rs/call vs ~25 rs international (15x cheaper)')
print('India Stack: Aadhaar eKYC $0.15, UPI 10B txns/mo, DigiLocker 676M users')


</details>
